In [0]:
# Install Required libraries

%pip install sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 138.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.6/196.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.9/542.9 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
dbutils.library.restartPython()

In [0]:
# Import Required Libraries

from pyspark.sql.functions import *
from pyspark.sql import Row
from datetime import datetime

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector search
import faiss
import numpy as np

# LLM
from transformers import pipeline

In [0]:
# Load Gold Layer Tables

# Load Gold Layer Tables
products_df = spark.table("globalmart.gold.dim_product")
vendors_df = spark.table("globalmart.gold.dim_vendor")

vendor_perf = spark.table("globalmart.gold.agg_vendor_performance")
slow_products = spark.table("globalmart.gold.agg_slow_moving_products")

# Enrich vendor
vendors_df = vendors_df.join(vendor_perf, "vendor_id", "left")

# Enrich product
products_df = products_df.join(
    slow_products.select("product_id").distinct().withColumn("is_slow_moving", lit(True)),
    "product_id",
    "left"
).fillna({"is_slow_moving": False})

In [0]:
# Create Documents


def product_to_doc(row):
    return f"""
    Product {row.get('product_name', 'unknown')} 
    belongs to category {row.get('categories', 'unknown')}.
    This product is {'slow-moving' if row.get('is_slow_moving') else 'fast-moving'}.
    """

def vendor_to_doc(row):
    return f"""
    Vendor {row.get('vendor_name', 'unknown')}.
    Their return rate is {row.get('return_rate', 'unknown')}.
    """

In [0]:
# Convert Data → List

# Convert small data (important for now)
product_rows = products_df.limit(1000).toPandas().to_dict(orient="records")
vendor_rows = vendors_df.limit(1000).toPandas().to_dict(orient="records")

# Create documents
product_docs = [product_to_doc(row) for row in product_rows]
vendor_docs = [vendor_to_doc(row) for row in vendor_rows]

documents = product_docs + vendor_docs

print("Total documents:", len(documents))





Total documents: 1007


In [0]:
# create embeddings

from sentence_transformers import SentenceTransformer

# Load local embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert documents into vectors
embeddings = model.encode(documents)

print("Embeddings created:", len(embeddings))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created: 1007


In [0]:
# Created FAISS Index

import faiss
import numpy as np

# Get vector dimension
dimension = embeddings.shape[1]

# Create index
index = faiss.IndexFlatL2(dimension)

# Add embeddings
index.add(np.array(embeddings))

print("FAISS index ready")

FAISS index ready


In [0]:
# Logging Table

def log_query(question, answer, docs):
    data = [Row(
        question=question,
        answer=answer,
        retrieved_docs=str(docs),
        timestamp=datetime.now()
    )]

    spark.createDataFrame(data).write.mode("append").saveAsTable(
        "globalmart.gold.rag_query_history"
    )

In [0]:
def ask(question):

    # Step 1: embedding
    q_emb = model.encode([question])

    # Step 2: search
    D, I = index.search(np.array(q_emb), 5)

    # Step 3: context
    docs = [documents[i] for i in I[0]]

    # remove duplicates 
    docs = list(dict.fromkeys(docs))

    # Step 4: take top 3
    docs = docs[:3]

    # Step 5: simple grounded answer (LLM-like)

    if "slow-moving" in question.lower():
        answer = "\n".join([
            doc for doc in docs if "slow-moving" in doc.lower()
        ])

    elif "fast-moving" in question.lower():
        answer = "\n".join([
            doc for doc in docs if "fast-moving" in doc.lower()
        ])

    elif "return rate" in question.lower():
        answer = "\n".join(docs)

    else:
        answer = "\n".join(docs)

    if answer.strip() == "":
        answer = "Answer not available in data."

    # Step 6: logging
    log_query(question, answer, docs)

    return answer

In [0]:
print(ask("Which products are slow-moving?"))


    Product Maurice Oiled Leather Boot 
    belongs to category Men,Shoes,Boots.
    This product is slow-moving.
    

    Product Faas 500 V4 Running Shoes 
    belongs to category Women,Shoes,Running,Neutral.
    This product is slow-moving.
    

    Product Jeromy Suede Sneaker 
    belongs to category Sneakers,Men,Shoes.
    This product is slow-moving.
    


In [0]:
questions = [
    "Which products are slow-moving?",
    "Which vendors have high return rate?",
    "List slow-moving products",
    "Which vendors perform poorly?",
    "What products are fast-moving?"
]

for q in questions:
    print("\nQ:", q)
    print("A:", ask(q))


Q: Which products are slow-moving?
A: 
    Product Maurice Oiled Leather Boot 
    belongs to category Men,Shoes,Boots.
    This product is slow-moving.
    

    Product Faas 500 V4 Running Shoes 
    belongs to category Women,Shoes,Running,Neutral.
    This product is slow-moving.
    

    Product Jeromy Suede Sneaker 
    belongs to category Sneakers,Men,Shoes.
    This product is slow-moving.
    

Q: Which vendors have high return rate?
A: 
    Vendor Pacific Trade Co..
    Their return rate is nan.
    

    Vendor Velocity Logistics.
    Their return rate is 0.26288659793814434.
    

    Vendor Johnsons Logistics.
    Their return rate is 0.3023255813953488.
    

Q: List slow-moving products
A: 
    Product Maurice Oiled Leather Boot 
    belongs to category Men,Shoes,Boots.
    This product is slow-moving.
    

    Product Bernie Mev Comfi 
    belongs to category Shoes,Clothing, Shoes & Jewelry,Women,Sandals,Platforms & Wedges.
    This product is slow-moving.
    

Q: Wh